In [2]:
import numpy as np
import nibabel as nib
from scipy.ndimage import label, generate_binary_structure, binary_dilation, find_objects, center_of_mass
import os

def advanced_smart_join(input_path, output_path, max_gap_pixels=2, area_ratio_thresh=0.25, abs_contact_thresh=1500):
    print(f"Loading prediction mask: {input_path}")
    
    # 1. Load Data
    nii_img = nib.load(input_path)
    pred_data = nii_img.get_fdata().astype(np.uint16)
    fixed_data = np.copy(pred_data)
    struct = generate_binary_structure(3, 3) 
    
    # 2. Extract initial fragments and Disc Barrier
    disc_mask = (pred_data == 3)
    odd_labels, num_odd = label(pred_data == 1, structure=struct)
    even_labels, num_even = label(pred_data == 2, structure=struct)
    
    initial_fragments = np.zeros_like(pred_data, dtype=np.uint32)
    initial_fragments[pred_data == 1] = odd_labels[pred_data == 1]
    initial_fragments[pred_data == 2] = even_labels[pred_data == 2] + num_odd
    
    total_frags = num_odd + num_even
    print(f"Initial isolated fragments: {total_frags}")
    if total_frags == 0: return

    # 3. Precompute properties (Volumes, Bounding Boxes, Centroids)
    volumes = {i: np.sum(initial_fragments == i) for i in range(1, total_frags + 1)}
    slices = find_objects(initial_fragments)
    centroids = {i: center_of_mass(initial_fragments == i) for i in range(1, total_frags + 1)}
    
    # --- SMART THOUGHT: Calculate 'Typical' Vertebra Volume dynamically ---
    # We take the median of the top 7 largest components (assuming at least 7 vertebrae in a scan)
    sorted_vols = sorted(volumes.values(), reverse=True)
    typical_vol = np.median(sorted_vols[:min(7, len(sorted_vols))])
    max_allowed_vol = 1.35 * typical_vol 
    print(f"Estimated Typical Vertebra Volume: {typical_vol:.0f} voxels")

    # Volume-Aware Union-Find
    parent = {i: i for i in range(1, total_frags + 1)}
    current_volumes = {i: volumes[i] for i in range(1, total_frags + 1)}
    
    def find(i):
        if parent[i] == i: return i
        parent[i] = find(parent[i])
        return parent[i]
        
    def union(i, j):
        root_i, root_j = find(i), find(j)
        if root_i != root_j:
            parent[root_i] = root_j
            current_volumes[root_j] += current_volumes[root_i] # Update the running total size

    def get_padded_slice(bbox, pad, shape):
        return tuple(slice(max(0, s.start - pad), min(dim, s.stop + pad)) for s, dim in zip(bbox, shape))

    # 4. Phase 1: Build the Graph of Potential Connections
    # print(f"Phase 1: Scanning for connections...")
    # potential_merges = [] # List of tuples: (contact_volume, overlap_ratio, i, j)
    
    # for i in range(1, total_frags + 1):
    #     if volumes[i] == 0 or slices[i-1] is None: continue
            
    #     crop_slice = get_padded_slice(slices[i-1], max_gap_pixels, pred_data.shape)
    #     crop_fragments = initial_fragments[crop_slice]
    #     crop_discs = disc_mask[crop_slice]
        
    #     frag_i_mask = (crop_fragments == i)
    #     dilated_i = binary_dilation(frag_i_mask, structure=struct, iterations=max_gap_pixels)
    #     restricted_dilation = dilated_i & (~crop_discs) # The Disc Barrier!
        
    #     overlapping_ids = np.unique(crop_fragments[restricted_dilation])
        
    #     for j in overlapping_ids:
    #         if j == 0 or j <= i: continue # Avoid double counting (i,j) and (j,i)
                
    #         contact_volume = np.sum(restricted_dilation & (crop_fragments == j))
    #         min_volume = min(volumes[i], volumes[j])
    #         overlap_ratio = contact_volume / min_volume
            
    #         is_noise = min_volume < (0.05 * typical_vol) # Fragments < 5% of typical size
            
    #         # If it meets any condition, add it to our candidate pool
    #         if (overlap_ratio > area_ratio_thresh) or (contact_volume > abs_contact_thresh) or (is_noise and overlap_ratio > 0.05):
    #             potential_merges.append((contact_volume, overlap_ratio, i, j))

    # # 5. Phase 2: Execute Smart Merges (Req 1 & Req 2)
    # # Sort descending by contact_volume, so we ALWAYS merge the strongest connections first
    # potential_merges.sort(key=lambda x: x[0], reverse=True)
    
    # merges_made = 0
    # for contact_vol, ratio, i, j in potential_merges:
    #     root_i, root_j = find(i), find(j)
    #     if root_i == root_j: continue
            
    #     proposed_volume = current_volumes[root_i] + current_volumes[root_j]
        
    #     # --- REQUIREMENT 2: Prevent Massive Merges ---
    #     if proposed_volume > max_allowed_vol:
    #         # ONLY block if BOTH components are substantial (e.g., > 20% of a typical bone).
    #         # This allows a massive 1.2x vertebra to still absorb a tiny 0.01x noise fragment.
    #         if current_volumes[root_i] > (0.2 * typical_vol) and current_volumes[root_j] > (0.2 * typical_vol):
    #             continue # Block this merge!
                
    #     # If safe, execute merge
    #     union(i, j)
    #     merges_made += 1

    # # --- STEP: IVD-BASED TOP VERTEBRA ISOLATION ---
    # disc_indices = np.where(pred_data == 3)
    # if disc_indices[2].size > 0:
    #     # Find the top boundary of the highest disc
    #     highest_disc_z_top = np.max(disc_indices[2])
    #     # We can also take a 'buffer' or use the midline of the highest disc
    #     # Let's use the top boundary as our "hard border"
    #     print(f"Landmark Found: Top-most Disc ends at Z-slice {highest_disc_z_top}")

    #     top_tier_fragments = []
        
    #     for i in range(1, total_frags + 1):
    #         if slices[i-1] is None: continue
            
    #         z_min, z_max = slices[i-1][2].start, slices[i-1][2].stop
    #         z_centroid = centroids[i][2]
            
    #         # Condition 1: Centroid is above the top disc
    #         is_above_disc = z_centroid > highest_disc_z_top
            
    #         # Condition 2: "Major Volume" check (More than 60% of bone is above the disc)
    #         # This handles fragments that might straddle the line
    #         total_h = max(1, z_max - z_min)
    #         above_fraction = max(0, z_max - highest_disc_z_top) / total_h
            
    #         if is_above_disc or above_fraction > 0.6:
    #             top_tier_fragments.append(i)
        
    #     if top_tier_fragments:
    #         print(f"Top-Tier Isolation: Grouping fragments {top_tier_fragments} as the Top Vertebra.")
    #         # Force-merge all these into a single "Super-Root"
    #         first_frag = top_tier_fragments[0]
    #         for other_frag in top_tier_fragments[1:]:
    #             union(first_frag, other_frag)
            
    #         # LOCKING MECHANISM:
    #         # We mark this root as 'finalized' so Phase 2 won't merge it downward
    #         finalized_roots = {find(first_frag)}
    #     else:
    #         finalized_roots = set()
    # else:
    #     finalized_roots = set()
    #     print("No IVD detected. Skipping Disc-based isolation.")

    # # 4. Phase 1: Build the Graph with Z-Axis Affinity Scoring
    # print(f"Phase 1: Scanning for connections and calculating Affinity Scores...")
    # potential_merges = [] 
    
    # for i in range(1, total_frags + 1):
    #     if volumes[i] == 0 or slices[i-1] is None: continue
            
    #     crop_slice = get_padded_slice(slices[i-1], max_gap_pixels, pred_data.shape)
    #     crop_fragments = initial_fragments[crop_slice]
    #     crop_discs = disc_mask[crop_slice]
        
    #     frag_i_mask = (crop_fragments == i)
    #     dilated_i = binary_dilation(frag_i_mask, structure=struct, iterations=max_gap_pixels)
    #     restricted_dilation = dilated_i & (~crop_discs) 
        
    #     overlapping_ids = np.unique(crop_fragments[restricted_dilation])
        
    #     for j in overlapping_ids:
    #         if j == 0 or j <= i: continue 
                
    #         contact_volume = np.sum(restricted_dilation & (crop_fragments == j))
    #         min_volume = min(volumes[i], volumes[j])
    #         overlap_ratio = contact_volume / min_volume
            
    #         is_noise = min_volume < (0.05 * typical_vol) 
            
    #         # --- THE NEW SMART LOGIC: Z-Overlap Fraction ---
    #         # Extract the vertical (Z) boundaries of both fragments
    #         z_min_i, z_max_i = slices[i-1][2].start, slices[i-1][2].stop
    #         z_min_j, z_max_j = slices[j-1][2].start, slices[j-1][2].stop
            
    #         # Calculate how many Z-slices they share
    #         overlap_start = max(z_min_i, z_min_j)
    #         overlap_end = min(z_max_i, z_max_j)
    #         z_overlap_pixels = max(0, overlap_end - overlap_start)
            
    #         # What percentage of fragment i's total height is covered by j?
    #         height_i = max(1, z_max_i - z_min_i)
    #         z_overlap_fraction = z_overlap_pixels / height_i
            
    #         # Create the final Merge Score: 
    #         # If they share vertical space, score stays high. 
    #         # If they only touch at the very tip (low Z-overlap), score gets crushed.
    #         merge_score = contact_volume * (0.1 + z_overlap_fraction)
            
    #         if (overlap_ratio > area_ratio_thresh) or (contact_volume > abs_contact_thresh) or (is_noise and overlap_ratio > 0.05):
    #             # Notice we are now appending the merge_score first!
    #             potential_merges.append((merge_score, contact_volume, overlap_ratio, i, j))

    # # 5. Phase 2: Execute Smart Merges
    # # Sort descending by MERGE_SCORE, not just raw contact volume!
    # # This guarantees the stripe will evaluate its connection to the gray vertebra BEFORE the green one.
    # potential_merges.sort(key=lambda x: x[0], reverse=True)
    
    # merges_made = 0
    # for merge_score, contact_vol, ratio, i, j in potential_merges:
    #     root_i, root_j = find(i), find(j)
    #     if root_i == root_j: continue

    #     # NEW PROTECTIVE RULE: 
    #     # If one root is the 'Top Tier' and the other is a main body below it, 
    #     # DO NOT allow the merge.
    #     if root_i in finalized_roots or root_j in finalized_roots:
    #         # Only allow if the other piece is also above the disc line
    #         # (Which shouldn't happen if we grouped correctly above)
    #         continue
            
    #     proposed_volume = current_volumes[root_i] + current_volumes[root_j]
        
    #     # Prevent Massive Merges 
    #     if proposed_volume > max_allowed_vol:
    #         if current_volumes[root_i] > (0.2 * typical_vol) and current_volumes[root_j] > (0.2 * typical_vol):
    #             continue 
                
    #     union(i, j)
    #     merges_made += 1


    # # 6. Phase 3: Orphan Rescue (Req 3)
    # # Find components that are still isolated and smaller than 30% of a typical bone
    # # all_roots = set(find(i) for i in range(1, total_frags + 1) if volumes[i] > 0)
    # # main_bodies = [r for r in all_roots if current_volumes[r] >= (0.3 * typical_vol)]
    # # orphans = [r for r in all_roots if current_volumes[r] < (0.3 * typical_vol)]
    
    # # print(f"Phase 3: Rescuing {len(orphans)} orphan fragments...")
    # # orphans_rescued = 0
    
    # # for orphan in orphans:
    # #     if not main_bodies: break
            
    # #     # Find the closest main body using 3D Euclidean distance between centroids
    # #     orphan_centroid = np.array(centroids[orphan])
        
    # #     best_match = None
    # #     min_distance = float('inf')
        
    # #     for main_body in main_bodies:
    # #         mb_centroid = np.array(centroids[main_body])
    # #         dist = np.linalg.norm(orphan_centroid - mb_centroid)
    # #         if dist < min_distance:
    # #             min_distance = dist
    # #             best_match = main_body
                
    # #     if best_match is not None:
    # #         # Force merge the orphan to its closest main body
    # #         parent[orphan] = best_match
    # #         current_volumes[best_match] += current_volumes[orphan]
    # #         orphans_rescued += 1

    # all_roots = set(find(i) for i in range(1, total_frags + 1) if volumes[i] > 0)
    # main_bodies = [r for r in all_roots if current_volumes[r] >= (0.3 * typical_vol)]
    # orphans = [r for r in all_roots if current_volumes[r] < (0.3 * typical_vol)]
    
    # print(f"Phase 3: Rescuing {len(orphans)} orphan fragments using Contact Area + Z-Distance...")
    # orphans_rescued = 0
    
    # for orphan in orphans:
    #     if not main_bodies: break
            
    #     # --- TIER 1: CONTACT AREA CHECK ---
    #     # Get a cropped region around the orphan to save memory
    #     crop_slice = get_padded_slice(slices[orphan-1], max_gap_pixels, pred_data.shape)
    #     crop_fragments = initial_fragments[crop_slice]
    #     crop_discs = disc_mask[crop_slice]
        
    #     crop_orphan = (crop_fragments == orphan)
    #     dilated_orphan = binary_dilation(crop_orphan, structure=struct, iterations=max_gap_pixels)
    #     restricted_dilation = dilated_orphan & (~crop_discs) # Respect the disc barrier
        
    #     # Calculate contact volume with surrounding components
    #     contact_vols = {}
    #     overlapping_labels = np.unique(crop_fragments[restricted_dilation])
        
    #     for lbl in overlapping_labels:
    #         if lbl == 0 or lbl == orphan: continue
    #         root_lbl = find(lbl)
    #         if root_lbl in main_bodies:
    #             # Count exactly how many voxels overlap
    #             overlap_voxels = np.sum((crop_fragments == lbl) & restricted_dilation)
    #             contact_vols[root_lbl] = contact_vols.get(root_lbl, 0) + overlap_voxels
        
    #     best_match = None
        
    #     if contact_vols:
    #         # The orphan physically touches at least one main body.
    #         # Select the one with the MAXIMUM contact surface area.
    #         best_match = max(contact_vols, key=contact_vols.get)
            
    #     else:
    #         # --- TIER 2: SPATIAL FALLBACK (Centroid + Z-Containment) ---
    #         # The orphan is completely floating. Find the best spatial fit.
    #         best_score = float('inf')
    #         orphan_centroid = np.array(centroids[orphan])
    #         orphan_z_min = slices[orphan-1][2].start
    #         orphan_z_max = slices[orphan-1][2].stop
            
    #         for main_body in main_bodies:
    #             mb_centroid = np.array(centroids[main_body])
    #             dist = np.linalg.norm(orphan_centroid - mb_centroid)
                
    #             mb_z_min = slices[main_body-1][2].start
    #             mb_z_max = slices[main_body-1][2].stop
                
    #             is_contained_z = (orphan_z_min >= mb_z_min) and (orphan_z_max <= mb_z_max)
                
    #             score = dist
    #             if is_contained_z:
    #                 score *= 0.1 # Massive discount if vertically aligned inside the host
                    
    #             if score < best_score:
    #                 best_score = score
    #                 best_match = main_body

    #     # Execute the rescue
    #     if best_match is not None:
    #         parent[orphan] = best_match
    #         current_volumes[best_match] += current_volumes[orphan]
    #         orphans_rescued += 1

    # --- STEP: IVD-BASED TOP VERTEBRA ISOLATION ---
    disc_indices = np.where(pred_data == 3)
    finalized_roots = set()
    highest_disc_z_top = -1

    if disc_indices[2].size > 0:
        highest_disc_z_top = np.max(disc_indices[2])
        print(f"highest_disc_z_top: {highest_disc_z_top}")
        top_tier_fragments = []
        
        for i in range(1, total_frags + 1):
            if slices[i-1] is None: continue
            z_max = slices[i-1][2].stop
            z_centroid = centroids[i][2]
            
            # If the fragment is mostly above the highest disc
            if z_centroid > highest_disc_z_top-10 or z_max > (highest_disc_z_top + 5):
                top_tier_fragments.append(i)
        
        if top_tier_fragments:
            first_frag = top_tier_fragments[0]
            for other_frag in top_tier_fragments[1:]:
                union(first_frag, other_frag)
            finalized_roots.add(find(first_frag))
            print(f"Top-Tier Isolated: Root {find(first_frag)} is now protected.")

    # ... [Keep Phase 1 & 2 as they are in your previous snippet] ...
    # 4. Phase 1: Build the Graph with Z-Axis Affinity Scoring
    print(f"Phase 1: Scanning for connections and calculating Affinity Scores...")
    potential_merges = [] 
    
    for i in range(1, total_frags + 1):
        if volumes[i] == 0 or slices[i-1] is None: continue
            
        crop_slice = get_padded_slice(slices[i-1], max_gap_pixels, pred_data.shape)
        crop_fragments = initial_fragments[crop_slice]
        crop_discs = disc_mask[crop_slice]
        
        frag_i_mask = (crop_fragments == i)
        dilated_i = binary_dilation(frag_i_mask, structure=struct, iterations=max_gap_pixels)
        restricted_dilation = dilated_i & (~crop_discs) 
        
        overlapping_ids = np.unique(crop_fragments[restricted_dilation])
        
        for j in overlapping_ids:
            if j == 0 or j <= i: continue 
                
            contact_volume = np.sum(restricted_dilation & (crop_fragments == j))
            min_volume = min(volumes[i], volumes[j])
            overlap_ratio = contact_volume / min_volume
            
            is_noise = min_volume < (0.05 * typical_vol) 
            
            # --- THE NEW SMART LOGIC: Z-Overlap Fraction ---
            # Extract the vertical (Z) boundaries of both fragments
            z_min_i, z_max_i = slices[i-1][2].start, slices[i-1][2].stop
            z_min_j, z_max_j = slices[j-1][2].start, slices[j-1][2].stop
            
            # Calculate how many Z-slices they share
            overlap_start = max(z_min_i, z_min_j)
            overlap_end = min(z_max_i, z_max_j)
            z_overlap_pixels = max(0, overlap_end - overlap_start)
            
            # What percentage of fragment i's total height is covered by j?
            height_i = max(1, z_max_i - z_min_i)
            z_overlap_fraction = z_overlap_pixels / height_i
            
            # Create the final Merge Score: 
            # If they share vertical space, score stays high. 
            # If they only touch at the very tip (low Z-overlap), score gets crushed.
            merge_score = contact_volume * (0.1 + z_overlap_fraction)
            
            if (overlap_ratio > area_ratio_thresh) or (contact_volume > abs_contact_thresh) or (is_noise and overlap_ratio > 0.05):
                # Notice we are now appending the merge_score first!
                potential_merges.append((merge_score, contact_volume, overlap_ratio, i, j))

    # 5. Phase 2: Execute Smart Merges
    # Sort descending by MERGE_SCORE, not just raw contact volume!
    # This guarantees the stripe will evaluate its connection to the gray vertebra BEFORE the green one.
    potential_merges.sort(key=lambda x: x[0], reverse=True)
    
    merges_made = 0
    for merge_score, contact_vol, ratio, i, j in potential_merges:
        root_i, root_j = find(i), find(j)
        if root_i == root_j: continue

        # NEW PROTECTIVE RULE: 
        # If one root is the 'Top Tier' and the other is a main body below it, 
        # DO NOT allow the merge.
        if root_i in finalized_roots or root_j in finalized_roots:
            # Only allow if the other piece is also above the disc line
            # (Which shouldn't happen if we grouped correctly above)
            continue
            
        proposed_volume = current_volumes[root_i] + current_volumes[root_j]
        
        # Prevent Massive Merges 
        if proposed_volume > max_allowed_vol:
            if current_volumes[root_i] > (0.2 * typical_vol) and current_volumes[root_j] > (0.2 * typical_vol):
                continue 
                
        union(i, j)
        merges_made += 1

    # 6. Phase 3: Smart Orphan Rescue
    all_roots = set(find(i) for i in range(1, total_frags + 1) if volumes[i] > 0)
    
    # CRITICAL FIX: The top-tier root is ALWAYS a main body, even if tiny
    main_bodies = [r for r in all_roots if current_volumes[r] >= (0.3 * typical_vol) or r in finalized_roots]
    # Orphans are small things that are NOT the top-tier
    orphans = [r for r in all_roots if current_volumes[r] < (0.3 * typical_vol) and r not in finalized_roots]
    
    print(f"Phase 3: Rescuing {len(orphans)} orphans. (Protected Top-Tier from being rescued)")
    orphans_rescued = 0
    
    for orphan in orphans:
        if not main_bodies: break
        
        orphan_centroid = np.array(centroids[orphan])
        o_z_min, o_z_max = slices[orphan-1][2].start, slices[orphan-1][2].stop
        
        best_match = None
        best_score = float('inf')
        
        for mb in main_bodies:
            # RULE: If the orphan is below the top disc, it cannot be rescued by the top vertebra
            if mb in finalized_roots and o_z_max < highest_disc_z_top:
                continue
                
            mb_centroid = np.array(centroids[mb])
            dist = np.linalg.norm(orphan_centroid - mb_centroid)
            
            # THE STRIPE FIX: Check Z-alignment
            mb_z_min, mb_z_max = slices[mb-1][2].start, slices[mb-1][2].stop
            
            # Does this orphan exist within the vertical span of this main body?
            is_contained_z = (o_z_min >= mb_z_min - 2) and (o_z_max <= mb_z_max + 2)
            
            score = dist
            if is_contained_z:
                score *= 0.05  # Huge priority boost for vertically aligned stripes
            
            if score < best_score:
                best_score = score
                best_match = mb

        if best_match is not None:
            parent[orphan] = best_match
            current_volumes[best_match] += current_volumes[orphan]
            orphans_rescued += 1

    # 7. Compile Final Instances
    final_instances = {}
    for i in range(1, total_frags + 1):
        if volumes[i] == 0: continue
        root = find(i)
        if root not in final_instances:
            final_instances[root] = []
        final_instances[root].append(i)
        
    num_final = len(final_instances)
    print(f"\n--- Component Counts ---")
    print(f"Total bone fragments BEFORE merging : {total_frags}")
    print(f"Primary Merges Executed           : {merges_made}")
    print(f"Orphans Force-Rescued             : {orphans_rescued}")
    print(f"Total unified instances AFTER     : {num_final}")
    print(f"------------------------\n")

    # 8. Apply Labels (Protect 3 and 4)
    fixed_data[(pred_data == 1) | (pred_data == 2)] = 0 
    
    current_instance_id = 101
    for root, fragment_list in final_instances.items():
        instance_mask = np.isin(initial_fragments, fragment_list)
        safe_to_write = instance_mask & (pred_data != 3) & (pred_data != 4)
        fixed_data[safe_to_write] = current_instance_id
        current_instance_id += 1

    # 9. Save
    fixed_img = nib.Nifti1Image(fixed_data, nii_img.affine, nii_img.header)
    nib.save(fixed_img, output_path)
    print(f"Saved highly robust instance mask to: {output_path}")

if __name__ == "__main__":
    INPUT_MASK = "../nnUNet_results/Dataset102_SpiderOddEven/predictions_2d_best_checkpoint/spider_048.nii.gz"      
    OUTPUT_MASK = "prediction_smart_instances_adv_048.nii.gz" 
    
    # We can safely use a wider gap because the Disc Barrier and Volume Caps protect us
    MAX_GAP = 1
    AREA_THRESH = 0.25
    ABS_CONTACT = 1500 
    
    if os.path.exists(INPUT_MASK):
        advanced_smart_join(INPUT_MASK, OUTPUT_MASK, 
                             max_gap_pixels=MAX_GAP, 
                             area_ratio_thresh=AREA_THRESH, 
                             abs_contact_thresh=ABS_CONTACT)
    else:
        print(f"Error: Could not find '{INPUT_MASK}'.")

Loading prediction mask: ../nnUNet_results/Dataset102_SpiderOddEven/predictions_2d_best_checkpoint/spider_048.nii.gz
Initial isolated fragments: 35
Estimated Typical Vertebra Volume: 38888 voxels
highest_disc_z_top: 474
Top-Tier Isolated: Root 7 is now protected.
Phase 1: Scanning for connections and calculating Affinity Scores...
Phase 3: Rescuing 1 orphans. (Protected Top-Tier from being rescued)

--- Component Counts ---
Total bone fragments BEFORE merging : 35
Primary Merges Executed           : 25
Orphans Force-Rescued             : 1
Total unified instances AFTER     : 8
------------------------

Saved highly robust instance mask to: prediction_smart_instances_adv_048.nii.gz


In [1]:
import numpy as np
import nibabel as nib
from scipy.ndimage import label, generate_binary_structure, center_of_mass

def dice_score(a, b):
    inter = np.sum(a & b)
    return 2.0 * inter / (a.sum() + b.sum() + 1e-8)


# --------------------------------------------------
# detect GT disc ordering axis
# --------------------------------------------------
def infer_ivd_axis(gt):

    ids = np.unique(gt)
    ids = ids[(ids >= 201) & (ids <= 299)]

    if len(ids) < 2:
        return 2, 1

    pts = []

    for gid in ids:
        c = center_of_mass(gt == gid)
        pts.append((gid, c))

    pts.sort(key=lambda t: t[0])

    coords = np.array([p[1] for p in pts])

    # check variation along each axis
    ranges = coords.max(axis=0) - coords.min(axis=0)
    axis = np.argmax(ranges)

    vals = coords[:, axis]
    trend = np.mean(np.diff(vals))

    direction = 1 if trend > 0 else -1

    return axis, direction


# --------------------------------------------------
# main evaluator
# --------------------------------------------------
def process_and_eval_ivds(pred_path, gt_path, typical_ivd_vol=5000):

    pred = nib.load(pred_path).get_fdata().astype(np.uint16)
    gt   = nib.load(gt_path).get_fdata().astype(np.uint16)

    struct = generate_binary_structure(3, 3)

    # predicted semantic discs = class 3
    comp, n = label(pred == 3, structure=struct)


    ivds = []

    for i in range(1, n + 1):

        mask = (comp == i)
        vol = mask.sum()

        if vol > 0.1 * typical_ivd_vol:

            c = center_of_mass(mask)

            ivds.append({
                "mask": mask,
                "centroid": c,
                "vol": vol
            })

    # --------------------------
    # infer correct ordering
    # --------------------------
    axis, direction = infer_ivd_axis(gt)

    ivds.sort(
        key=lambda d: direction * d["centroid"][axis]
    )

    # --------------------------
    # evaluate
    # --------------------------
    dices = []

    for k, inst in enumerate(ivds):

        label_id = 201 + k

        g = (gt == label_id)
        p = inst["mask"]

        d = dice_score(p, g)
        dices.append(d)

    mean_dice = np.mean(dices) if dices else 0

    print("\nMean IVD Dice =", round(mean_dice, 4))

    return mean_dice

mean_dice = process_and_eval_ivds(
    "prediction_smart_instances_adv_009.nii.gz",
    "../nnUNet_raw/Dataset102_SpiderOddEven/labelsTs_instanceGT/Spider_009.nii.gz"
    )


Mean IVD Dice = 0.8319


In [ ]:
import os
import glob
import numpy as np
import nibabel as nib

from scipy.ndimage import (
    label,
    generate_binary_structure,
    binary_dilation,
    find_objects,
    center_of_mass
)

# ==========================================================
# CONFIG
# ==========================================================

GT_FOLDER = "../nnUNet_raw/Dataset102_SpiderOddEven/labelsTs_instanceGT"
PRED_FOLDER = "../nnUNet_results/Dataset102_SpiderOddEven/predictions_2d_1/"
OUT_FOLDER = "./instance_predictions"

os.makedirs(OUT_FOLDER, exist_ok=True)

def advanced_smart_join_v2(
    input_path,
    output_path,
    max_gap_pixels=1,
    area_ratio_thresh=0.20,
    abs_contact_thresh=1200,
    expected_min_count=4,
    expected_max_count=10
):
    print(f"Loading prediction mask: {input_path}")

    # ------------------------------------------------------
    # 1. LOAD
    # ------------------------------------------------------
    nii = nib.load(input_path)
    pred = nii.get_fdata().astype(np.uint16)
    fixed = pred.copy()

    struct = generate_binary_structure(3, 3)

    bone_mask = (pred == 1) | (pred == 2)
    disc_mask = (pred == 3)
    canal_mask = (pred == 4)

    # ------------------------------------------------------
    # 2. INITIAL COMPONENTS
    # keep odd/even separated first
    # ------------------------------------------------------
    odd_lab, n_odd = label(pred == 1, structure=struct)
    even_lab, n_even = label(pred == 2, structure=struct)

    frags = np.zeros_like(pred, dtype=np.uint32)

    frags[pred == 1] = odd_lab[pred == 1]
    frags[pred == 2] = even_lab[pred == 2] + n_odd

    total = n_odd + n_even
    print(f"Initial fragments: {total}")

    if total == 0:
        print("No fragments found.")
        return

    # ------------------------------------------------------
    # 3. PROPERTIES
    # ------------------------------------------------------
    volumes = {}
    boxes = find_objects(frags)
    centroids = {}

    for i in range(1, total + 1):
        mask = (frags == i)
        v = int(mask.sum())
        volumes[i] = v

        if v > 0:
            centroids[i] = np.array(center_of_mass(mask), dtype=np.float64)

    # dynamic size prior
    sorted_vols = sorted([v for v in volumes.values() if v > 0], reverse=True)
    topk = sorted_vols[:min(7, len(sorted_vols))]
    typical_vol = np.median(topk) if len(topk) else 1000.0

    max_allowed_vol = 1.40 * typical_vol
    orphan_thresh = 0.25 * typical_vol
    noise_thresh = 0.05 * typical_vol

    print(f"Typical vertebra volume: {typical_vol:.1f}")

    # ------------------------------------------------------
    # 4. ESTIMATE SPINE AXIS USING PCA OF CENTROIDS
    # removes dependence on z-axis assumptions
    # ------------------------------------------------------
    pts = np.array([centroids[i] for i in centroids.keys()])

    if len(pts) >= 2:
        mean_pt = pts.mean(axis=0)
        X = pts - mean_pt
        cov = np.dot(X.T, X) / max(1, len(X))
        vals, vecs = np.linalg.eigh(cov)
        spine_axis = vecs[:, np.argmax(vals)]
    else:
        mean_pt = pts[0]
        spine_axis = np.array([0, 0, 1], dtype=np.float64)

    # project all fragments onto spine axis
    projections = {}
    for i in centroids:
        projections[i] = float(np.dot(centroids[i] - mean_pt, spine_axis))

    # ------------------------------------------------------
    # 5. UNION FIND
    # ------------------------------------------------------
    parent = {i: i for i in range(1, total + 1)}
    current_vol = {i: volumes[i] for i in range(1, total + 1)}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False

        parent[ra] = rb
        current_vol[rb] += current_vol[ra]
        return True

    # ------------------------------------------------------
    # 6. BUILD CANDIDATE MERGES
    # ------------------------------------------------------
    def padded_slice(b, pad):
        out = []
        for sl, dim in zip(b, pred.shape):
            out.append(slice(max(0, sl.start - pad), min(dim, sl.stop + pad)))
        return tuple(out)

    candidates = []

    print("Scanning candidate merges...")

    for i in range(1, total + 1):

        if volumes[i] == 0 or boxes[i - 1] is None:
            continue

        crop = padded_slice(boxes[i - 1], max_gap_pixels)

        crop_frag = frags[crop]
        crop_disc = disc_mask[crop]

        mi = (crop_frag == i)

        di = binary_dilation(
            mi,
            structure=struct,
            iterations=max_gap_pixels
        )

        # disc barrier
        reachable = di & (~crop_disc)

        neigh = np.unique(crop_frag[reachable])

        for j in neigh:

            if j == 0 or j <= i:
                continue

            mj = (crop_frag == j)

            contact = int(np.sum(reachable & mj))
            if contact == 0:
                continue

            minv = max(1, min(volumes[i], volumes[j]))
            overlap_ratio = contact / minv

            # distance along spine axis
            axis_dist = abs(projections[i] - projections[j])

            # Euclidean centroid distance
            dist = np.linalg.norm(centroids[i] - centroids[j])

            # parity bonus (odd-even expected to merge more often)
            parity_bonus = 1.0
            if (i <= n_odd and j > n_odd) or (i > n_odd and j <= n_odd):
                parity_bonus = 1.15

            is_noise = min(volumes[i], volumes[j]) < noise_thresh

            if (
                overlap_ratio > area_ratio_thresh
                or contact > abs_contact_thresh
                or (is_noise and overlap_ratio > 0.05)
            ):
                # higher score = stronger merge
                score = (
                    contact * parity_bonus
                    + 250.0 * overlap_ratio
                    - 8.0 * axis_dist
                    - 0.25 * dist
                )

                candidates.append(
                    (score, i, j, contact, overlap_ratio, axis_dist)
                )

    # ------------------------------------------------------
    # 7. EXECUTE MERGES GREEDILY
    # ------------------------------------------------------
    candidates.sort(reverse=True, key=lambda x: x[0])

    merges = 0

    for score, i, j, contact, ratio, axis_dist in candidates:

        ri, rj = find(i), find(j)

        if ri == rj:
            continue

        proposed = current_vol[ri] + current_vol[rj]

        # prevent huge false merges unless one side tiny
        if proposed > max_allowed_vol:
            if current_vol[ri] > 0.20 * typical_vol and current_vol[rj] > 0.20 * typical_vol:
                continue

        # fragments too far apart along axis unlikely same vertebra
        if axis_dist > 3.5:
            if min(current_vol[ri], current_vol[rj]) > noise_thresh:
                continue

        if union(i, j):
            merges += 1

    print(f"Primary merges: {merges}")

    # ------------------------------------------------------
    # 8. ROOTS AFTER MERGE
    # ------------------------------------------------------
    roots = {}
    for i in range(1, total + 1):
        if volumes[i] == 0:
            continue
        r = find(i)
        roots.setdefault(r, []).append(i)

    # recompute root centroid / projection
    root_vol = {}
    root_cent = {}
    root_proj = {}

    for r, members in roots.items():
        vv = sum(volumes[m] for m in members)
        cc = sum(centroids[m] * volumes[m] for m in members) / max(vv, 1)

        root_vol[r] = vv
        root_cent[r] = cc
        root_proj[r] = float(np.dot(cc - mean_pt, spine_axis))

    # --- NEW: LANDMARK DETECTION ---
    disc_indices = np.where(pred == 3)
    highest_disc_z = -1
    if disc_indices[2].size > 0:
        # Assuming Z is the last dimension (pred_data.shape[2])
        highest_disc_z = np.max(disc_indices[2])
        print(f"Top-most Disc Landmark: Z = {highest_disc_z}")

    # ------------------------------------------------------
    # 9. ORPHAN RESCUE
    # ------------------------------------------------------
    # mains = [r for r in roots if root_vol[r] >= orphan_thresh]
    # orphans = [r for r in roots if root_vol[r] < orphan_thresh]

    mains = []
    orphans = []

    for r in roots:
        # Calculate if this root is 'High Altitude'
        is_above_disc = root_cent[r][2] > highest_disc_z
        
        if root_vol[r] >= orphan_thresh or is_above_disc:
            mains.append(r)
        else:
            orphans.append(r)

    print(f"Phase 3: {len(mains)} Main bodies, {len(orphans)} Orphans. (Top vertebra protected)")

    rescued = 0

    for o in orphans:

        if not mains:
            break

        best = None
        best_score = 1e18

        for m in mains:

            if o == m:
                continue

            d = np.linalg.norm(root_cent[o] - root_cent[m])
            ad = abs(root_proj[o] - root_proj[m])

            score = d + 3.0 * ad

            if score < best_score:
                best_score = score
                best = m

        if best is not None:
            parent[o] = best
            root_vol[best] += root_vol[o]
            rescued += 1

    print(f"Orphans rescued: {rescued}")

    # ------------------------------------------------------
    # 10. FINAL INSTANCES
    # ------------------------------------------------------
    final = {}

    for i in range(1, total + 1):
        if volumes[i] == 0:
            continue

        r = find(i)
        final.setdefault(r, []).append(i)

    # If too many final instances, keep largest plausible ones
    final_items = list(final.items())

    final_items.sort(
        key=lambda kv: sum(volumes[x] for x in kv[1]),
        reverse=True
    )

    if len(final_items) > expected_max_count:
        final_items = final_items[:expected_max_count]

    print(f"Final instance count: {len(final_items)}")

    # ------------------------------------------------------
    # 11. ORDER ALONG SPINE AXIS (bottom-up for SPIDER)
    # SPIDER label 1 = lowest visible lumbar vertebra
    # ------------------------------------------------------
    ordered = []

    for root, members in final_items:
        vox = sum(volumes[m] for m in members)
        cc = sum(centroids[m] * volumes[m] for m in members) / max(vox, 1)
        proj = float(np.dot(cc - mean_pt, spine_axis))
        ordered.append((root, members, proj))

    # bottom-most first = smallest or largest depends axis sign.
    # choose direction so more labels trend with x if possible.
    ordered.sort(key=lambda x: x[2])

    # ------------------------------------------------------
    # 12. WRITE OUTPUT
    # ------------------------------------------------------
    fixed[(pred == 1) | (pred == 2)] = 0

    current_label = 101

    for root, members, proj in ordered:

        mask = np.isin(frags, members)
        safe = mask & (~disc_mask) & (~canal_mask)

        fixed[safe] = current_label
        current_label += 1

    out = nib.Nifti1Image(fixed, nii.affine, nii.header)
    nib.save(out, output_path)


# ==========================================================
# DICE EVALUATION
# ==========================================================

def dice_score(a, b):
    inter = np.sum(a & b)
    return 2.0 * inter / (a.sum() + b.sum() + 1e-8)


def relabel_prediction_bottom_up(pred):

    pred_ids = np.unique(pred)
    pred_ids = pred_ids[pred_ids >= 101]

    if len(pred_ids) == 0:
        return np.zeros_like(pred, dtype=np.int32)

    objs = []

    for pid in pred_ids:
        z, y, x = center_of_mass(pred == pid)
        objs.append((pid, x))   # SPIDER sagittal commonly ordered along x

    objs.sort(key=lambda t: t[1])

    new_pred = np.zeros_like(pred, dtype=np.int32)

    for new_label, (pid, _) in enumerate(objs, start=1):
        new_pred[pred == pid] = new_label

    return new_pred


def evaluate_spider(pred_path, gt_path):

    pred = nib.load(pred_path).get_fdata().astype(np.int32)
    gt = nib.load(gt_path).get_fdata().astype(np.int32)

    gt = np.where((gt >= 1) & (gt <= 9), gt, 0)

    pred_relabeled = relabel_prediction_bottom_up(pred)

    gt_ids = np.unique(gt)
    gt_ids = gt_ids[gt_ids > 0]

    dices = []

    for lab in gt_ids:
        g = (gt == lab)
        p = (pred_relabeled == lab)

        d = dice_score(p, g)
        dices.append(d)

    mean_dice = float(np.mean(dices)) if len(dices) else 0.0
    return mean_dice, dices


# ==========================================================
# MAIN PIPELINE
# ==========================================================

def run_full_pipeline():

    pred_files = sorted(glob.glob(os.path.join(PRED_FOLDER, "*.nii.gz")))

    all_scores = []

    print("=" * 70)
    print("RUNNING INSTANCE SEGMENTATION + DICE EVALUATION")
    print("=" * 70)

    for pred_file in pred_files:

        base = os.path.basename(pred_file)         # spider_045.nii.gz
        num = base.split("_")[1].split(".")[0]    # 045

        gt_file = os.path.join(
            GT_FOLDER,
            f"Spider_{num}.nii.gz"
        )

        if not os.path.exists(gt_file):
            print(f"GT missing for {base}")
            continue

        out_file = os.path.join(
            OUT_FOLDER,
            f"prediction_instances_{num}.nii.gz"
        )

        # Step 1: semantic -> instance
        # advanced_smart_join_v2(
        #     pred_file,
        #     out_file
        # )

        MAX_GAP = 1
        AREA_THRESH = 0.25
        ABS_CONTACT = 1500 
        
        advanced_smart_join(pred_file, out_file, 
                            max_gap_pixels=MAX_GAP, 
                            area_ratio_thresh=AREA_THRESH, 
                            abs_contact_thresh=ABS_CONTACT)

        # Step 2: dice
        mean_dice, per_label = evaluate_spider(
            out_file,
            gt_file
        )

        all_scores.append(mean_dice)

        print(f"{base:20s}  Mean Dice = {mean_dice:.4f}")

    print("\n" + "=" * 70)

    if len(all_scores):
        print(f"Total Cases      : {len(all_scores)}")
        print(f"Dataset Avg Dice : {np.mean(all_scores):.4f}")
        print(f"Dataset Std Dice : {np.std(all_scores):.4f}")
        print(f"Best Dice        : {np.max(all_scores):.4f}")
        print(f"Worst Dice       : {np.min(all_scores):.4f}")
    else:
        print("No valid cases found.")

    print("=" * 70)


# ==========================================================
# RUN
# ==========================================================

if __name__ == "__main__":
    run_full_pipeline()